HW1 (Foundations - Ch. 2–3):
Normalization & tokenization: select a data set of your choice, conduct a brief ex-
ploratory data analysis, decide an arbitrary task you are supposed to implement, and
apply the text normalization and tokenization as discussed in the classed.
Implement either Edit Distance or Byte-pairing algorithm and apply it to you data set
and analysis the output and comment on it.

In [1]:
import re
from typing import List
import zipfile
import os
import pandas as pd
from itertools import combinations
from tqdm import tqdm

First, download the dataset

In [2]:
#!/bin/bash
!curl -L -o ~/Downloads/imdb-dataset-of-50k-movie-reviews.zip\
  https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 25.7M  100 25.7M    0     0  5257k      0  0:00:05  0:00:05 --:--:-- 6408k0  4632k      0  0:00:05  0:00:03  0:00:02 6089k


In [3]:
os.makedirs("dataset_imdb", exist_ok=True)


In [4]:
zip_path = os.path.expanduser('~/Downloads/imdb-dataset-of-50k-movie-reviews.zip')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('dataset_imdb')

In [5]:
imdb_data = pd.read_csv("dataset_imdb/IMDB Dataset.csv")

Create a text from different sentences

In [6]:
text = " ".join(imdb_data['review'].fillna('').astype(str))

In [7]:
def tokenize(text: str, pattern: str | None = None) -> List[str]:
    text = text.lower()
    pattern = r"(?:\w+')\w+|(?:[A-Z]\.)+|\w+(?:-\w+)*|[\w+\.]"
    tokens = re.findall(pattern, text)
    return tokens

Tokenize it now:

In [8]:
tokenized_data = tokenize(text)

implement a BPE algo 

In [9]:
def bpe(data, count_total: int):
    freq = dict()
    vocab = set()

    for word in data:
        word = "_" + word
        vocab.update(word)
        freq[word] = freq.get(word, 0) + 1

    words = {tuple(word): count for word, count in freq.items()}

    for k in tqdm(range(count_total)):
        tmp_dict = dict()
        for symb, count in words.items():
            for i in range(len(symb) - 1):
                pair = (symb[i], symb[i + 1])
                tmp_dict[pair] = tmp_dict.get(pair, 0) + count

        max_pair = max(tmp_dict, key=tmp_dict.get)
        new_words = dict()
        for symb, count in words.items():
            i = 0
            new_symb = list()
            while i < len(symb):
                if i < len(symb) - 1 and (symb[i], symb[i + 1]) == max_pair:
                    new_symb.append(symb[i] + symb[i + 1])
                    i += 2
                else:
                    new_symb.append(symb[i])
                    i += 1
            new_words[tuple(new_symb)] = count

        vocab.add(max_pair[0] + max_pair[1])
        words = new_words

    return vocab

In [10]:
len_unique = len(set(tokenized_data[:12000]))
print(f'Unique size of vocab (without BPE): {len_unique}')

Unique size of vocab (without BPE): 3034


In [11]:
len_stast = len(set(char for word in tokenized_data[:25000] for char in word))
print(f'Vocab. size in the 0 iter of BPE: {len_stast}')

Vocab. size in the 0 iter of BPE: 45


Run BPE algorighm:

In [13]:
bpe_result = bpe(tokenized_data[:25000], 300)
print(f'Vocab. size in the 300 iter of BPE: {len(bpe_result)}')
print('examples:')
len(bpe_result)
for word in bpe_result:
    if len(word) > 2:
        print(word)

100%|██████████| 300/300 [00:01<00:00, 259.92it/s]

Vocab. size in the 300 iter of BPE: 346
examples:
ers
_film
act
_he
out
_which
itt
_even
_int
ion
_st
_do
ple
ort
_on
_dis
hing
ist
_ro
ver
res
_other
_fil
_watch
_good
_out
_how
_can
ars
ood
_when
ain
urn
_only
_di
one
ate
and
her
_was
_but
ell
ther
_them
_get
_it's
_my
_in
ant
_vi
ful
_the
_see
_off
ack
_sh
est
_sp
ust
ith
_con
_have
_wat
_de
_him
_su
ing
_some
ong
_i'
_would
_as
ich
_his
_it
end
_co
_mu
_go
_for
_le
_char
ice
ook
_if
ake
ous
ity
_her
_more
_from
_story
ies
_pl
are
ber
_like
_you
ess
ive
_me
_no
_act
_all
_what
_of
_are
ind
_to
very
_or
red
n't
ent
able
reat
_se
ough
_charact
_ne
_wh
_sc
_th
ard
ation
_just
_fr
_ex
_pro
_tr
_up
man
ght
ight
_about
_any
_so
_who
ame
_at
_lo
_movie
_ar
all
_com
_that
own
ory
ble
_an
_man
_is
_this
_with
gin
_li
ill
ence
_ha
_scen
_br
ven
_be
_re
_movi
_and
ings
_not
_cl
ould
_en
_one
_they
_has
ere
ound
ome
_bec
ure
ull
art
_we
_us
ast
_wor
_ch
_than
ving
_al
_un
_tim
_there
ally
_ab
_mo
ter
_by
